# 03 — Training Results Analysis

Compare fine-tuning results across all three models:
- **TinyLlama 1.1B** + LoRA
- **Qwen2.5 1.5B** + LoRA  
- **Llama 3.1 8B** + QLoRA

Metrics tracked:
- Training loss curve
- Evaluation perplexity
- BLEU and ROUGE-L on test set
- Trainable parameter count

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# ----------------------------------------------------------------
# Load training metrics from logs (produced by MetricsLoggerCallback)
# Replace these paths with your actual training output directories
# ----------------------------------------------------------------

LOG_FILES = {
    'TinyLlama-1.1B': 'outputs/logs/tinyllama_metrics.jsonl',
    'Qwen2.5-1.5B': 'outputs/logs/qwen25_metrics.jsonl',
    'Llama3.1-8B-QLoRA': 'outputs/logs/llama31_metrics.jsonl',
}

def load_metrics(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        return pd.DataFrame()  # Return empty if not trained yet
    records = [json.loads(line) for line in p.read_text().splitlines() if line.strip()]
    return pd.DataFrame(records)

metrics = {name: load_metrics(path) for name, path in LOG_FILES.items()}

for name, df in metrics.items():
    status = f'{len(df)} eval checkpoints' if not df.empty else 'Not trained yet'
    print(f'{name}: {status}')

In [ ]:
# Loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'TinyLlama-1.1B': 'steelblue', 'Qwen2.5-1.5B': 'coral', 'Llama3.1-8B-QLoRA': 'green'}

for name, df in metrics.items():
    if df.empty:
        continue
    if 'eval_loss' in df.columns:
        axes[0].plot(df['step'], df['eval_loss'], label=name, color=colors.get(name))
    if 'eval_perplexity' in df.columns:
        axes[1].plot(df['step'], df['eval_perplexity'], label=name, color=colors.get(name))

axes[0].set_title('Eval Loss per Step')
axes[0].set_xlabel('Training Step')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].set_title('Eval Perplexity per Step')
axes[1].set_xlabel('Training Step')
axes[1].set_ylabel('Perplexity')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Final results summary table
# Populate after running evaluate.py for each model
EVAL_RESULTS = {
    'TinyLlama-1.1B': {
        'Final Eval Loss': '-',
        'Perplexity': '-',
        'BLEU': '-',
        'ROUGE-L': '-',
        'Trainable Params': '~41M (3.7%)',
        'Training Time': '-',
        'GPU VRAM': '~4 GB',
    },
    'Qwen2.5-1.5B': {
        'Final Eval Loss': '-',
        'Perplexity': '-',
        'BLEU': '-',
        'ROUGE-L': '-',
        'Trainable Params': '~56M (3.5%)',
        'Training Time': '-',
        'GPU VRAM': '~6 GB',
    },
    'Llama3.1-8B-QLoRA': {
        'Final Eval Loss': '-',
        'Perplexity': '-',
        'BLEU': '-',
        'ROUGE-L': '-',
        'Trainable Params': '~168M (2.0%)',
        'Training Time': '-',
        'GPU VRAM': '~8 GB (4-bit)',
    },
}

results_df = pd.DataFrame(EVAL_RESULTS).T
print('=== Final Evaluation Results ===')
results_df